# astrowidget — Interactive Sky Viewer

Replaces ipyaladin for OVRO-LWA visualization. No FITS round-trip — zarr → numpy → GPU directly.

**What this replaces:**
| ipyaladin workflow | astrowidget workflow |
|---|---|
| Extract WCS from zarr attrs | Same |
| Build `astropy.io.fits.HDUList` from numpy | **Eliminated** |
| Serialize FITS to bytes | **Eliminated** |
| `aladin.add_fits(hdul)` | `widget.set_image(array, wcs)` |
| Broken sphere overlay | Working sphere overlay |

## Setup

Launch from the ovro-lwa-portal pixi environment:
```
pixi run jupyter lab
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

import os
import ovro_lwa_portal as ovro

## 1. Load OVRO-LWA data from OSN

Same `open_dataset()` call as the interactive_viz_demo notebook.

In [ ]:
# DOI for the 10t x 10f x 4096x4096 test dataset
DOI = "10.33569/4q7nb-ahq31"

ds = ovro.open_dataset(
    DOI,
    production=False,
    storage_options={
        "key": os.environ["OSN_KEY"],
        "secret": os.environ["OSN_SECRET"],
    },
)

print(f"Loaded: {DOI}")
print(f"  Dims:  {dict(ds.dims)}")
print(f"  Vars:  {list(ds.data_vars)}")
print(f"  Freq:  {float(ds.frequency.values[0])/1e6:.1f} – {float(ds.frequency.values[-1])/1e6:.1f} MHz")
print(f"  WCS:   {ds.radport.has_wcs}")

## 2. Display on the celestial sphere — one line

**Old way (ipyaladin):** Extract WCS, build FITS HDU, serialize, `aladin.add_fits()` — 20+ lines.

**New way:** `widget.set_dataset(ds)` — reads zarr, caches slices, extracts WCS, renders. One line.

In [ ]:
from astrowidget import SkyWidget
from astropy.coordinates import SkyCoord
import astropy.units as u

widget = SkyWidget()

# One line: zarr → cached slices → WCS → GPU. No FITS anywhere.
widget.set_dataset(ds)

# Navigate to phase center with full hemisphere
from astrowidget import get_wcs
wcs = get_wcs(ds)
phase_center = SkyCoord(ra=wcs.wcs.crval[0], dec=wcs.wcs.crval[1], unit="deg", frame="fk5")
widget.goto(phase_center, fov=180 * u.deg)

print(f"Phase center: {phase_center.to_string('hmsdms')}")
print(f"vmin={widget.vmin:.2f}, vmax={widget.vmax:.2f}")
widget

## 4. Navigate to known radio sources

Use `goto()` with any `SkyCoord` to navigate the sphere.

In [ ]:
# Navigate to Cassiopeia A — brightest radio source in the northern sky
cas_a = SkyCoord.from_name("Cas A")
widget.goto(cas_a, fov=30 * u.deg)
print(f"Navigated to Cas A: {cas_a.to_string('hmsdms')}")

In [ ]:
# Navigate to Cygnus A
cyg_a = SkyCoord.from_name("Cyg A")
widget.goto(cyg_a, fov=20 * u.deg)
print(f"Navigated to Cyg A: {cyg_a.to_string('hmsdms')}")

In [ ]:
# Back to full hemisphere
widget.goto(phase_center, fov=180 * u.deg)

## 5. Change display settings from Python

Colormap, stretch, and color scale are GPU-only updates — instant, no data re-upload.

In [ ]:
# Colormap, stretch, grid — all instant GPU updates
widget.colormap = "viridis"
widget.stretch = "sqrt"
widget.auto_scale(percentile_low=5, percentile_high=95)
print(f"Grid: {widget.show_grid}")

# Toggle grid off/on
# widget.show_grid = False

In [ ]:
# Switch slices via traitlets — observer auto-updates the display
widget.time_idx = 5
widget.freq_idx = 5
widget.colormap = "inferno"
widget.stretch = "linear"
print(f"Showing time={widget.time_idx}, freq={widget.freq_idx} ({widget._cube.freq_mhz[5]:.1f} MHz)")

# Try other slices:
# widget.time_idx = 0
# widget.freq_idx = 9

## 6. Read view state from Python

After panning/zooming in the widget, read back the current view position.

In [ ]:
# Read current view (updates after pan/zoom interaction)
print(f"View center: RA={widget.view_ra:.4f}°, Dec={widget.view_dec:.4f}°")
print(f"Field of view: {widget.view_fov:.2f}°")
print(f"Colormap: {widget.colormap}, Stretch: {widget.stretch}")
print(f"Color scale: [{widget.vmin:.2f}, {widget.vmax:.2f}]")

## 7. Overlay radio data on DSS survey tiles

`widget.overlay("DSS")` stacks the SkyWidget on top of Aladin Lite.
The radio image is transparent where there's no data, so the optical sky shows through.

In [ ]:
# Set background BEFORE displaying — Aladin loads in the same container
widget2 = SkyWidget()
widget2.set_dataset(ds)  # auto-navigates to phase center with fitted FOV
widget2.background_survey = "DSS"
widget2

In [ ]:
# Switch survey
# widget2.background_survey = "Mellinger"
# widget2.background_survey = "WISE"
# widget2.background_survey = ""  # disable